### Ablation study

The goal of this ablation study is to measure the impact of each feature.

I will use the 'best' hyperparameters obtained from the prediction step for each model, and try removing one feature at a time from the dataset and see how performance changees.  

In [1]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt

Load dataset

In [2]:
import pandas as pd
import numpy as np

DATASET_FILEPATH = '../data/dataset.csv'
df = pd.read_csv(DATASET_FILEPATH)

In [4]:
# Define features and target
target = 'price'
features = [
    'make',
    'model',
    'year',
    'body_type',
    'mileage',
    'color', 
    'fuel_type',
    'engine_capacity',
    'engine_power',
    'gearbox', 
    'transmission',
    'pollution_standard'
]

numerical_features = [
    'year',
    'mileage',
    'engine_capacity',
    'engine_power'
]

categorical_features = [
    'make',
    'model',
    'body_type',
    'color',
    'fuel_type',
    'gearbox',
    'transmission',
    'pollution_standard'
]

X = df[features]
y = df[target]

Train test split (stratified by price bins)

In [5]:
# stratified split by price bins
df['price_bin'] = pd.qcut(df['price'], q=5, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=df['price_bin'],
    random_state=17
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

Train set: 1514 samples
Test set: 379 samples


Use the best parameters from the gridsearch from the prediction step (hardcoded here sadly)

In [8]:
rf_hyperparams = {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}

xgb_hyperparams = {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 300, 'subsample': 0.9}

In [11]:
rf_base_model = RandomForestRegressor(**rf_hyperparams)

xgb_base_model = xgb.XGBRegressor(**xgb_hyperparams)

models = {
    "Random forest": rf_base_model,
    "XGBoost": xgb_base_model
}

In [24]:
def get_pipeline(model, excluded_feature=None):
    # filter features
    current_numerical = [f for f in numerical_features if f != excluded_feature]
    current_categorical = [f for f in categorical_features if f != excluded_feature]
    
    # build transformers
    transformers = []
    transformers.append(('num', 'passthrough', current_numerical))
    transformers.append(('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), current_categorical))
    
    preprocessor = ColumnTransformer(transformers=transformers)
    
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

In [25]:
def compute_metrics(y_test, y_pred):
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return r2, mae, rmse

In [ ]:
results = {}

for model_name, model in models.items():
    results[model_name] = []

    # iterate through all features and exclude it, then compute metrics
    for feature in features:
        print(f"Removing {feature}...")

        pipeline = get_pipeline(model, excluded_feature=feature)

        # train and evaluate
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        r2, mae, rmse = compute_metrics(y_test, y_pred)
        
        results[model_name].append({
            "Feature removed": feature,
            "R2": r2,
            "MAE": mae,
            "RMSE": rmse 
        })

Removing make...
Removing model...
Removing year...
Removing body_type...
Removing mileage...
Removing color...
Removing fuel_type...
Removing engine_capacity...
Removing engine_power...
Removing gearbox...
Removing transmission...
Removing pollution_standard...
Removing make...
Removing model...
Removing year...
Removing body_type...
Removing mileage...
Removing color...
Removing fuel_type...
Removing engine_capacity...
Removing engine_power...
Removing gearbox...
Removing transmission...
Removing pollution_standard...


In [38]:
results["XGBoost"] = results["XGBoost"][1:]

In [62]:
rf_results = pd.DataFrame(results["Random forest"])

xgb_results = pd.DataFrame(results["XGBoost"])

In [63]:
rf_results

,Feature removed,R2,MAE,RMSE
0,make,0.843541,3693.823768,6343.446042
1,model,0.822530,3851.169835,6755.961569
2,year,0.818027,4280.670188,6841.136487
3,body_type,0.841575,3759.001182,6383.163264
4,mileage,0.839346,3737.887682,6427.918613
5,color,0.846611,3656.311974,6280.891944
6,fuel_type,0.849953,3611.356159,6212.100020
7,engine_capacity,0.828070,3870.365032,6649.662789
8,engine_power,0.903878,3243.997434,4972.054639
9,gearbox,0.843246,3693.386232,6349.408503


In [64]:
xgb_results.style.hide()

Feature removed,R2,MAE,RMSE
make,0.875982,3265.380619,5647.634156
model,0.867147,3362.339292,5845.334594
year,0.844893,3804.047116,6315.975405
body_type,0.886015,3168.674335,5414.378611
mileage,0.880502,3305.030906,5543.763335
color,0.883478,3107.209894,5474.286706
fuel_type,0.891956,3068.908448,5271.381083
engine_capacity,0.888961,3271.125396,5343.941871
engine_power,0.908877,2950.973524,4841.024711
gearbox,0.884557,3201.304304,5448.889987


In [ ]:
# for latex table formatting: (ignore this)

for row in xgb_results.itertuples():
    print(f"{row[1]}\t & {row[2]:.3f} & {int(row[3])} & {int(row[4])} \\")

make	 & 0.876 & 3265 & 5647
model	 & 0.867 & 3362 & 5845
year	 & 0.845 & 3804 & 6315
body_type	 & 0.886 & 3168 & 5414
mileage	 & 0.881 & 3305 & 5543
color	 & 0.883 & 3107 & 5474
fuel_type	 & 0.892 & 3068 & 5271
engine_capacity	 & 0.889 & 3271 & 5343
engine_power	 & 0.909 & 2950 & 4841
gearbox	 & 0.885 & 3201 & 5448
transmission	 & 0.883 & 3184 & 5478
pollution_standard	 & 0.886 & 3216 & 5413


In [61]:
rf_results['Model'] = "Random forest"
xgb_results['Model'] = "XGBoost"

results_df = pd.concat([rf_results, xgb_results], ignore_index=True).pivot(index="Feature removed", columns='Model')
results_df

R2                     MAE               \
Model              Random forest   XGBoost Random forest      XGBoost   
Feature removed                                                         
body_type               0.841575  0.886015   3759.001182  3168.674335   
color                   0.846611  0.883478   3656.311974  3107.209894   
engine_capacity         0.828070  0.888961   3870.365032  3271.125396   
engine_power            0.903878  0.908877   3243.997434  2950.973524   
fuel_type               0.849953  0.891956   3611.356159  3068.908448   
gearbox                 0.843246  0.884557   3693.386232  3201.304304   
make                    0.843541  0.875982   3693.823768  3265.380619   
mileage                 0.839346  0.880502   3737.887682  3305.030906   
model                   0.822530  0.867147   3851.169835  3362.339292   
pollution_standard      0.843698  0.886048   3701.889961  3216.057064   
transmission            0.842618  0.883287   3731.866782  3184.919602   
year                    0.818027  0.844893   4280.670188  3804.047116   

                            RMSE               
Model              Random forest      XGBoost  
Feature removed                                
body_type            6383.163264  5414.378611  
color                6280.891944  5474.286706  
engine_capacity      6649.662789  5343.941871  
engine_power         4972.054639  4841.024711  
fuel_type            6212.100020  5271.381083  
gearbox              6349.408503  5448.889987  
make                 6343.446042  5647.634156  
mileage              6427.918613  5543.763335  
model                6755.961569  5845.334594  
pollution_standard   6340.242811  5413.578681  
transmission         6362.122900  5478.787563  
year                 6841.136487  6315.975405